In [ ]:
from theia.simulation.scenario_import import ScenarioFactory
from theia.simulation.theia_logging import LogLoader


loader = LogLoader(path="result.json")
with open("ballistic_missile_detection.scenario.json", "r") as file:
    scenario = ScenarioFactory.model_validate_json(file.read())
ballistic_missile = scenario.red_orbat.ballistic_missiles[0]

In [ ]:
fig, ax = ballistic_missile.plot_trajectory()

fig.tight_layout()

In [ ]:
ballistic_missile._times

In [ ]:
from theia.coordinates import CoordinateTransformations


path = [
    CoordinateTransformations.cartesian_to_geodetic(
        *s.state_vector.flatten()[[0, 2, 4]]
    )
    for s in loader.red_target_ground_truth[4].states
]

In [ ]:
detection = loader.blue_monostatic_radar_detections[0]
assert detection.target.id == 4

In [ ]:
import numpy as np


np.rad2deg(detection.elevation_angle)

In [ ]:
from theia.plotting import plot_profile


plot_profile(detection.radar.receiver.point, detection.target.point, "Sensor", "Target")

In [ ]:
from theia.grids import LatLonHeightGrid, LatLonTerrainGrid
from theia.visualization_3d import visualize_hbv_tree

terrain = scenario.terrain_model.to_terrain()

grid = LatLonTerrainGrid(
    lat_start=46.8602,
    lat_stop=47.6099,
    lat_res=0.001,
    lon_start=7.1933,
    lon_stop=8.3606,
    lon_res=0.001,
    terrain_model=terrain,
)

# fig = visualize_hbv_tree(
#     grid,
#     terrain.tree,
#     12,
#     vertical_exaggeration=10,
#     los_points=(detection.radar.receiver.point, detection.target.point),
# )
# fig.write_html("terrain.html")

In [ ]:
detection.target.point

In [ ]:
import datetime

from theia.export_paraview import ParaviewExporter, PointOfInterest


exporter = ParaviewExporter(
    ".",
    grid.lat_start,
    grid.lat_stop,
    grid.lat_res,
    grid.lon_start,
    grid.lon_stop,
    grid.lon_res,
    terrain,
    elevation_factor=10,
)
exporter.export(
    trajectories=[
        ballistic_missile.get_trajectory(True).select_subset(
            t_min=detection.time - datetime.timedelta(seconds=10),
        )
    ],
    pois=[
        PointOfInterest(
            id=0,
            label="Sensor",
            type="Rx",
            lat=detection.radar.receiver.lat,
            lon=detection.radar.receiver.lon,
            alt=detection.radar.receiver.alt,
        ),
        PointOfInterest(
            id=1,
            label="Target",
            type="Tx",
            lat=detection.target.lat,
            lon=detection.target.lon,
            alt=detection.target.alt,
        ),
    ],
)

In [ ]:
t_max = ballistic_missile.get_trajectory(True).times[-1]
t_max

In [ ]:
detection.time

In [ ]:
t_max - detection.time

In [ ]:
from theia.distance import line_of_sight_distance


line_of_sight_distance(*detection.radar.receiver.point.as_tuple(), *detection.target.point.as_tuple()) / 1000

In [ ]:
import folium

m = folium.Map((50.6808, 13.0298), control_scale=True, zoom_start=6)
folium.LatLngPopup().add_to(m)
folium.PolyLine([(p[0], p[1]) for p in path]).add_to(m)
folium.Marker((detection.target.lat, detection.target.lon)).add_to(m)
m